# Pista A — Motor empírico de simulación (ECO | Wind)

Sandbox de la **Fase 1 / Pista A** del plan técnico (`plan-tecnico-eco-wind.md`).
Objetivo: un pipeline trazable `simular(lat, lon, altura_buje, modelo, N) -> kWh_anual`
usando el catálogo de Flower Turbines ya validado (`engine/flower_turbines_curves.py`).

**Nota sobre entornos de ejecución:** este notebook se escribió y probó en un sandbox de
Claude Code sin salida de red a `power.larc.nasa.gov` (política de egress del entorno).
La celda de NASA POWER está armada para degradar sola a datos sintéticos cuando eso pasa,
así que el notebook corre de punta a punta en cualquier entorno — pero el resultado real
(con viento real del sitio) solo sale corriéndolo en **Google Colab** o cualquier entorno
con internet normal. Los pasos 3 y 4 (corrección de altura, ensamblado) sí están validados
acá con datos sintéticos.


## Paso 1 — Módulo base (`engine/flower_turbines_curves.py`)

In [1]:
import sys
sys.path.insert(0, "..")

import calendar
import numpy as np
import pandas as pd
import requests

from engine.flower_turbines_curves import (
    CURVE_COEFFICIENTS,
    power_isolated,
    bouquet_multiplier,
    power_in_bouquet,
)

print("Modelos disponibles:", list(CURVE_COEFFICIENTS))
print(f"Medium Tulip @ 12 m/s, aislada: {float(power_isolated(12, 'medium_tulip')):.1f} W")


Modelos disponibles: ['small_tulip', 'medium_tulip', 'three_m_tulip', 'large_tulip', 'al13_2m', 'al13_4m', 'al13_6m', 'al13_8m']
Medium Tulip @ 12 m/s, aislada: 622.2 W


## Paso 2 — Ingesta climática (NASA POWER Hourly)

Punto `community=SB`, parámetros `WS10M`/`WS50M`/`T2M`, formato JSON, año completo
(8,760 h ó 8,784 h en bisiesto).


In [2]:
NASA_POWER_HOURLY_URL = "https://power.larc.nasa.gov/api/temporal/hourly/point"


def fetch_nasa_power_hourly(lat, lon, year, community="SB",
                             parameters=("WS10M", "WS50M", "T2M")):
    """
    Descarga viento/temperatura horarios de NASA POWER para un año completo
    en una coordenada arbitraria. Devuelve un DataFrame con índice datetime
    horario y una columna por parámetro.

    NO EJECUTADO CON ÉXITO EN EL SANDBOX DE DESARROLLO: sin salida de red a
    power.larc.nasa.gov ahí. Validar en Colab (o cualquier entorno con
    internet normal) antes de confiar en el resultado.
    """
    params = {
        "parameters": ",".join(parameters),
        "community": community,
        "longitude": lon,
        "latitude": lat,
        "start": f"{year}0101",
        "end": f"{year}1231",
        "format": "JSON",
    }
    resp = requests.get(NASA_POWER_HOURLY_URL, params=params, timeout=60)
    resp.raise_for_status()
    param_data = resp.json()["properties"]["parameter"]
    df = pd.DataFrame(param_data)
    df.index = pd.to_datetime(df.index, format="%Y%m%d%H")
    df.index.name = "datetime"

    horas_esperadas = 8784 if calendar.isleap(year) else 8760
    if len(df) != horas_esperadas:
        raise ValueError(f"Esperaba {horas_esperadas} horas, llegaron {len(df)}")
    return df


In [3]:
def generar_clima_sintetico(year=2023, v_media=3.5, seed=42):
    """
    SOLO PARA PROBAR EL PIPELINE SIN RED. Serie horaria sintética con forma
    estacional simple (pico ilustrativo dic-abr, tipo estación seca de Costa
    Rica) + ruido Weibull. v_media=3.5 m/s es un valor ilustrativo, NO viene
    de ninguna fuente medida -- reemplazar por fetch_nasa_power_hourly() con
    dato real antes de sacar cualquier conclusión de negocio.
    """
    n_horas = 8784 if calendar.isleap(year) else 8760
    idx = pd.date_range(f"{year}-01-01", periods=n_horas, freq="h")
    rng = np.random.default_rng(seed)
    estacional = 1.0 + 0.4 * np.cos(2 * np.pi * (idx.dayofyear - 30) / 365)
    ruido = rng.weibull(2.0, size=n_horas)
    ws10m = np.clip(v_media * estacional * ruido / ruido.mean(), 0, None)
    return pd.DataFrame(
        {"WS10M": ws10m, "WS50M": ws10m * 1.15, "T2M": 22.0}, index=idx
    )


In [4]:
# Coordenada de prueba: San José, Valle Central, Costa Rica
LAT, LON, YEAR = 9.9281, -84.0907, 2023

try:
    df_clima = fetch_nasa_power_hourly(LAT, LON, YEAR)
    print(f"OK -- {len(df_clima)} horas reales descargadas de NASA POWER.")
    datos_reales = True
except Exception as exc:
    print(f"No se pudo descargar de NASA POWER en este entorno ({exc!r}).")
    print("Sigo con datos SINTÉTICOS solo para probar el resto del pipeline.")
    df_clima = generar_clima_sintetico(YEAR, v_media=3.5)
    datos_reales = False

df_clima.head()


No se pudo descargar de NASA POWER en este entorno (ProxyError(MaxRetryError("HTTPSConnectionPool(host='power.larc.nasa.gov', port=443): Max retries exceeded with url: /api/temporal/hourly/point?parameters=WS10M%2CWS50M%2CT2M&community=SB&longitude=-84.0907&latitude=9.9281&start=20230101&end=20231231&format=JSON (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))).
Sigo con datos SINTÉTICOS solo para probar el resto del pipeline.


,WS10M,WS50M,T2M
2023-01-01 00:00:00,8.325665,9.574514,22.0
2023-01-01 01:00:00,8.207046,9.438103,22.0
2023-01-01 02:00:00,8.291923,9.535712,22.0
2023-01-01 03:00:00,2.840222,3.266255,22.0
2023-01-01 04:00:00,1.578642,1.815438,22.0


## Paso 3 — Corrección de altura (perfil logarítmico)

In [5]:
def wind_at_height(v_ref, h_ref, h_target, z0=0.3):
    """
    Perfil logarítmico de viento: v(h) = v_ref * ln(h_target/z0) / ln(h_ref/z0)

    h_ref   : altura del dato de referencia (10 para WS10M, 50 para WS50M)
    h_target: altura real de buje de la turbina
    z0      : longitud de rugosidad (m). Default 0.3 = suburbano/urbano bajo.
              Ajustar por sitio: 0.03 campo abierto, 0.1 cultivos bajos,
              0.3 suburbano (default), 1.0 urbano denso.

    Las turbinas Flower Turbines son muy bajas (buje entre 1 y 6 m), casi
    siempre POR DEBAJO de los 10 m de referencia -- a diferencia de una HAWT
    grande (buje 80m+), acá la corrección casi siempre REDUCE la velocidad
    respecto al dato crudo de NASA POWER.
    """
    v_ref = np.asarray(v_ref, dtype=float)
    return v_ref * np.log(h_target / z0) / np.log(h_ref / z0)


# Autotest con valores sintéticos (no depende de la red)
print("Autotest perfil logarítmico, v_10m = 5.00 m/s:")
for h in [1.4, 3.0, 6.0, 10.0]:
    print(f"  altura buje={h:4.1f} m  ->  v={wind_at_height(5.0, 10, h):.2f} m/s")
assert abs(wind_at_height(5.0, 10, 10) - 5.0) < 1e-9, "en h_target=h_ref debe dar v_ref exacto"
print("OK: en h_target = h_ref da v_ref exacto.")


Autotest perfil logarítmico, v_10m = 5.00 m/s:
  altura buje= 1.4 m  ->  v=2.20 m/s
  altura buje= 3.0 m  ->  v=3.28 m/s
  altura buje= 6.0 m  ->  v=4.27 m/s
  altura buje=10.0 m  ->  v=5.00 m/s
OK: en h_target = h_ref da v_ref exacto.


## Paso 4 — Ensamblar `simular(lat, lon, altura_buje, modelo, N) -> kWh_anual`

In [6]:
def simular(df_clima, altura_buje, modelo, N, h_ref=10, z0=0.3, metodo_bouquet="real"):
    """
    Ensambla la serie horaria de potencia del clúster y la agrega a kWh
    mensual/anual, usando P(v) = k*v^3 x M(N) del motor empírico.

    df_clima: DataFrame con índice datetime horario y columna 'WS10M' (m/s)
              -- viene de fetch_nasa_power_hourly() o de datos sintéticos;
              la función no sabe ni le importa el origen.
    """
    v_hub = wind_at_height(df_clima["WS10M"].values, h_ref, altura_buje, z0=z0)
    potencia_w_por_turbina = power_in_bouquet(v_hub, modelo, N, metodo=metodo_bouquet)

    serie = pd.Series(potencia_w_por_turbina, index=df_clima.index,
                       name="potencia_W_por_turbina")
    energia_cluster_kwh = serie * N / 1000.0
    return {
        "serie_horaria_W_por_turbina": serie,
        "kwh_mensual": energia_cluster_kwh.resample("MS").sum(),
        "kwh_anual": float(energia_cluster_kwh.sum()),
    }


resultado = simular(df_clima, altura_buje=3.0, modelo="medium_tulip", N=3)
etiqueta = "REAL (NASA POWER)" if datos_reales else "SINTÉTICO -- no es dato real del sitio"
print(f"Escenario: Medium Tulip x3 (bouquet), buje a 3.0 m, San José CR, {YEAR} [{etiqueta}]")
print(f"kWh/año: {resultado['kwh_anual']:.1f}")
print()
print(resultado["kwh_mensual"])


Escenario: Medium Tulip x3 (bouquet), buje a 3.0 m, San José CR, 2023 [SINTÉTICO -- no es dato real del sitio]
kWh/año: 426.6

2023-01-01    78.248672
2023-02-01    69.182152
2023-03-01    64.546197
2023-04-01    38.412973
2023-05-01    21.077526
2023-06-01     9.701907
2023-07-01     6.164773
2023-08-01     6.898122
2023-09-01    10.271701
2023-10-01    20.338143
2023-11-01    36.439241
2023-12-01    65.307598
Freq: MS, Name: potencia_W_por_turbina, dtype: float64


## Paso 5 — Sanity check de orden de magnitud

In [7]:
# Referencia externa (Kilowatts UK, distribuidor UK): 1,000-5,000 kWh/año
# típico para turbinas pequeñas. No es para copiar el número -- Costa Rica
# tiene otro recurso eólico y contexto -- solo para confirmar que el orden
# de magnitud del resultado no es absurdo.
REF_UK_BAJO, REF_UK_ALTO = 1000, 5000
kwh = resultado["kwh_anual"]

print(f"Resultado [{etiqueta}]: {kwh:.0f} kWh/año")
print(f"Referencia UK (orden de magnitud, turbinas pequeñas): {REF_UK_BAJO}-{REF_UK_ALTO} kWh/año")

if REF_UK_BAJO * 0.3 <= kwh <= REF_UK_ALTO * 3:
    print("-> Orden de magnitud razonable.")
else:
    print("-> FUERA de rango incluso con margen amplio -- revisar el pipeline antes de confiar en él.")

if not datos_reales:
    print()
    print("!! Este sanity check corrió sobre datos SINTÉTICOS. Repetir en Colab con NASA POWER")
    print("   real antes de sacar cualquier conclusión sobre el recurso eólico de Costa Rica.")


Resultado [SINTÉTICO -- no es dato real del sitio]: 427 kWh/año
Referencia UK (orden de magnitud, turbinas pequeñas): 1000-5000 kWh/año
-> Orden de magnitud razonable.

!! Este sanity check corrió sobre datos SINTÉTICOS. Repetir en Colab con NASA POWER
   real antes de sacar cualquier conclusión sobre el recurso eólico de Costa Rica.


## Resumen y próximos pasos

- **Validado en este sandbox:** `engine/flower_turbines_curves.py` (Paso 1), corrección de
  altura por perfil logarítmico (Paso 3), y el ensamblado completo `simular()` (Paso 4) —
  con datos sintéticos.
- **Pendiente de validar con datos reales:** la celda de NASA POWER (Paso 2) — este sandbox
  no tiene salida a `power.larc.nasa.gov`. Abrir este notebook en Colab y volver a correr
  todas las celdas: si hay internet normal, `datos_reales` da `True` automáticamente y el
  resto del notebook usa viento real sin cambiar nada de código.
- **Variables abiertas** (sección 4 del plan, no bloqueantes): calibración K(v) contra datos
  de campo reales, z0 real del sitio (acá se usó 0.3 como default ilustrativo), validez del
  M(N) exponencial más allá de N=10 o en layouts 2D.
